In [1]:
%pip install torch transformers accelerate

Note: you may need to restart the kernel to use updated packages.


In [2]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-3B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

c:\Users\qalid\miniconda3\envs\ai-infra-learning\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
text = "I am sure this project"

tokens = tokenizer.tokenize(text) # makes it into tokens 
token_ids = tokenizer.encode(text)

print("tokens:", tokens)
print("token ids:", token_ids)

tokens: ['I', 'Ġam', 'Ġsure', 'Ġthis', 'Ġproject']
token ids: [40, 1079, 2704, 419, 2390]


In [5]:
import torch
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(model_name)
model.eval()

input_ids = torch.tensor([token_ids])

embedding_layer = model.model.embed_tokens
embeddings = embedding_layer(input_ids)

print("Embedding tensor shape:", embeddings.shape)
print()
print("First token embedding:")
print(embeddings[0, 0])

Loading weights: 100%|██████████| 434/434 [00:00<00:00, 2238.89it/s]


Embedding tensor shape: torch.Size([1, 5, 2048])

First token embedding:
tensor([-0.0065,  0.0243, -0.0103,  ...,  0.0204, -0.0255,  0.0103],
       dtype=torch.bfloat16, grad_fn=<SelectBackward0>)


In [6]:
layer0 = model.model.layers[0]
attention = layer0.self_attn

print("Wq:", attention.q_proj.weight.shape)
print("Wk:", attention.k_proj.weight.shape)
print("Wv:", attention.v_proj.weight.shape)

Wq: torch.Size([2048, 2048])
Wk: torch.Size([256, 2048])
Wv: torch.Size([256, 2048])


In [7]:
Q = attention.q_proj(embeddings)
K = attention.k_proj(embeddings)
V = attention.v_proj(embeddings)

print("Q shape:", Q.shape)
print("K shape:", K.shape)
print("V shape:", V.shape)

Q shape: torch.Size([1, 5, 2048])
K shape: torch.Size([1, 5, 256])
V shape: torch.Size([1, 5, 256])


# we take 3 inptuts of different lengths , tokenize, then pad them together to see if it would help

In [4]:
# First get the 3 values
a = "Hello my name is qalid"
b = "small text"
c = "The FitnessGram Pacer Test is a multistage aerobic capacity test that progressively gets more difficult as it continues. The 20 meter pacer test will begin in 30 seconds"


In [5]:
# now from text we need to essentially convert tokenes
aToken = tokenizer.tokenize(a)
bToken = tokenizer.tokenize(b)
cToken = tokenizer.tokenize(c)
# print the tokens for each
print(aToken)
print(bToken)
print(cToken)

['Hello', 'Ġmy', 'Ġname', 'Ġis', 'Ġq', 'al', 'id']
['small', 'Ġtext']
['The', 'ĠFitness', 'Gram', 'ĠP', 'acer', 'ĠTest', 'Ġis', 'Ġa', 'Ġmult', 'ist', 'age', 'Ġaerobic', 'Ġcapacity', 'Ġtest', 'Ġthat', 'Ġprogressively', 'Ġgets', 'Ġmore', 'Ġdifficult', 'Ġas', 'Ġit', 'Ġcontinues', '.', 'ĠThe', 'Ġ', '2', '0', 'Ġmeter', 'Ġp', 'acer', 'Ġtest', 'Ġwill', 'Ġbegin', 'Ġin', 'Ġ', '3', '0', 'Ġseconds']


In [ ]:
# now from tokens we need to convert to ids